# 05 — Comparación, candidatos y vecinos cercanos

In [ ]:
from pathlib import Path
import torch, json, hashlib, shutil, zipfile, sys
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display
from torch import nn
from torch.utils.data import Dataset, DataLoader
assert torch.cuda.is_available(), "Activa T4 GPU."
DEVICE = torch.device("cuda")
from google.colab import drive
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/PuertoPrisma")
OPTIONS = [("base_bce",15),("base_bce",30),("loss_label_smoothing",15),
           ("loss_label_smoothing",30),("stability_lr_d",15),("stability_lr_d",30)]
for option, (run, epoch) in enumerate(OPTIONS,1):
    folder = PROJECT / "runs" / run
    assert (folder / f"checkpoint_epoch_{epoch:03d}.pt").exists(), f"Falta checkpoint: {run}, época {epoch}"
    print(f"OPCIÓN {option}: {run}, época {epoch}")
    with Image.open(folder / f"fixed_epoch_{epoch:03d}.png") as im:
        display(im.resize((512,512),Image.Resampling.NEAREST))
print("Compara nitidez, coherencia del rostro, artefactos y variedad entre los 16 vectores.")

## Elegir checkpoint

In [ ]:
choice = int(input("Elige checkpoint: opción 1, 2, 3, 4, 5 o 6: "))
assert 1 <= choice <= len(OPTIONS)
reason = input("¿Por qué elegiste esa opción? Calidad, variedad y limitaciones: ").strip()
assert reason, "Debemos documentar la elección."
run_name, epoch = OPTIONS[choice-1]
CHECKPOINT = PROJECT / "runs" / run_name / f"checkpoint_epoch_{epoch:03d}.pt"
OUT = PROJECT / "candidates" / f"{run_name}_epoch_{epoch:03d}_seed_7070"
OUT.mkdir(parents=True,exist_ok=True)
print("Elegido:", run_name, "época", epoch)
print("Resultados:", OUT)

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, features=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, features*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(features*8), nn.ReLU(True),
            nn.ConvTranspose2d(features*8, features*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features*4), nn.ReLU(True),
            nn.ConvTranspose2d(features*4, features*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features*2), nn.ReLU(True),
            nn.ConvTranspose2d(features*2, features, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features), nn.ReLU(True),
            nn.ConvTranspose2d(features, 3, 4, 2, 1, bias=False), nn.Tanh())
    def forward(self, z):
        return self.net(z)


# Cargar exclusivamente los pesos de nuestros propios entrenamientos.
local = Path("/content/puerto_prisma_gallery")
local.mkdir(exist_ok=True)
local_checkpoint = local / "chosen_checkpoint.pt"
shutil.copy2(CHECKPOINT,local_checkpoint)
checkpoint_sha256 = hashlib.sha256(local_checkpoint.read_bytes()).hexdigest()
state = torch.load(local_checkpoint,map_location="cpu",weights_only=False)
config = state["config"]
G = Generator(config["latent_dim"],config["features"]).to(DEVICE)
G.load_state_dict(state["generator"])
G.eval()
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
POOL_SEED, COUNT = 7070, 200
identity = dict(run=run_name,epoch=epoch,checkpoint_sha256=checkpoint_sha256,
                pool_seed=POOL_SEED,count=COUNT,generation_batch_size=32)
metadata_file = OUT / "pool.json"
if metadata_file.exists():
    meta = json.loads(metadata_file.read_text())
    assert meta["identity"] == identity, "Los parámetros de este lote difieren; no sobrescribir."
else:
    meta = dict(identity=identity,checkpoint=str(CHECKPOINT),choice_reason=reason,
                torch_version=torch.__version__,latent_dim=config["latent_dim"],
                features=config["features"],source="Google LLC, Cartoon Set, CC BY 4.0",
                generated_count=COUNT,selected_count=0,selection_status="pending")
    metadata_file.write_text(json.dumps(meta,indent=2))
(OUT / "candidates").mkdir(exist_ok=True)
(OUT / "sheets").mkdir(exist_ok=True)
if (OUT / "noise.pt").exists():
    z = torch.load(OUT / "noise.pt",map_location="cpu",weights_only=True)
else:
    rng = torch.Generator().manual_seed(POOL_SEED)
    z = torch.randn(COUNT,config["latent_dim"],1,1,generator=rng)
    torch.save(z,OUT / "noise.pt")
assert z.shape == (COUNT,config["latent_dim"],1,1)
# Siempre el mismo lote; reutiliza los PNG si ya existen.
if not all((OUT / "candidates" / f"candidate_{i:03d}.png").exists() for i in range(1,COUNT+1)):
    with torch.inference_mode():
        for start in range(0,COUNT,32):
            samples = ((G(z[start:start+32].to(DEVICE)).cpu().clamp(-1,1)+1)*127.5).round().byte()
            for offset,sample in enumerate(samples):
                Image.fromarray(sample.permute(1,2,0).numpy()).save(OUT / "candidates" / f"candidate_{start+offset+1:03d}.png")
# Guardar solo los pesos del generador elegido, para regenerar la galería posteriormente.
if not (OUT / "generator.pt").exists():
    torch.save(dict(config=config,generator=state["generator"],checkpoint_sha256=checkpoint_sha256),OUT / "generator.pt")
del state
print("200 candidatos guardados, junto con sus vectores exactos y pesos.")

In [ ]:
def contact_sheet(ids, destination):
    sheet = Image.new("RGB",(8*144,7*160),"#eeeeee")
    draw = ImageDraw.Draw(sheet)
    for position, candidate_id in enumerate(ids):
        x,y = (position%8)*144,(position//8)*160
        with Image.open(OUT / "candidates" / f"candidate_{candidate_id:03d}.png") as im:
            sheet.paste(im.resize((128,128),Image.Resampling.NEAREST),(x+8,y+4))
        draw.text((x+8,y+136),f"ID {candidate_id:03d}",fill="black")
    sheet.save(destination)
    return sheet
for start in range(1,COUNT+1,50):
    sheet = contact_sheet(range(start,min(start+50,COUNT+1)),OUT / "sheets" / f"candidates_{start:03d}.png")
    display(sheet)
print("No cierres aún: sigue la búsqueda de vecinos cercanos.")

## Vecinos cercanos: características de CNN

In [ ]:
import torchvision
from torchvision.models import resnet18, ResNet18_Weights
from torch.nn import functional as F
archive = PROJECT / "cartoonset64.zip"
local_zip = local / "cartoonset64.zip"
shutil.copy2(archive,local_zip)
assert hashlib.sha256(local_zip.read_bytes()).hexdigest() == meta.get("data_sha256", json.loads((PROJECT / "runs" / run_name / "config.json").read_text())["data_sha256"])
DATA = local / "real"
with zipfile.ZipFile(local_zip) as bundle:
    assert all(not Path(n).is_absolute() and ".." not in Path(n).parts for n in bundle.namelist())
    if not DATA.exists():
        DATA.mkdir()
        bundle.extractall(DATA)
real_files = sorted(DATA.rglob("*.png"))
assert len(real_files)==10000
candidate_files = [OUT / "candidates" / f"candidate_{i:03d}.png" for i in range(1,COUNT+1)]
weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
encoder = resnet18(weights=weights)
encoder.fc = nn.Identity()
encoder = encoder.to(DEVICE).eval()
class FeatureDataset(Dataset):
    def __init__(self, paths): self.paths=paths
    def __len__(self): return len(self.paths)
    def __getitem__(self,index):
        with Image.open(self.paths[index]) as im:
            return preprocess(im.convert("RGB"))
def embed(paths):
    features=[]
    loader = DataLoader(FeatureDataset(paths),batch_size=64,shuffle=False,num_workers=2,pin_memory=True)
    with torch.inference_mode():
        for batch_id,batch in enumerate(loader):
            features.append(F.normalize(encoder(batch.to(DEVICE,non_blocking=True)),dim=1).cpu())
            if (batch_id+1)%25==0:
                print(f"Características: {min((batch_id+1)*64,len(paths))}/{len(paths)}",flush=True)
    return torch.cat(features)
print("Buscando en 10000 imágenes reales...")
real_features=embed(real_files)
fake_features=embed(candidate_files)
similarities=fake_features @ real_features.T
best_similarity, best_index = similarities.max(dim=1)
records=[]
for i in range(COUNT):
    neighbor=real_files[best_index[i].item()]
    records.append(dict(candidate_id=i+1,candidate_file=f"candidates/candidate_{i+1:03d}.png",
                        nearest_training_file=str(neighbor.relative_to(DATA)),
                        cosine_similarity=float(best_similarity[i]),
                        cosine_distance=float(1-best_similarity[i])))
(OUT / "nearest_neighbors.json").write_text(json.dumps(records,indent=2))
meta["neighbors_method"] = dict(model="ResNet18",weights="IMAGENET1K_V1",feature_dim=512,
                                  metric="cosine_distance",training_images=10000,
                                  torchvision_version=torchvision.__version__,
                                  limitation="ImageNet domain differs from cartoons; visual inspection required")
meta["data_sha256"]=hashlib.sha256(local_zip.read_bytes()).hexdigest()
metadata_file.write_text(json.dumps(meta,indent=2))
print("Vecinos calculados y documentados. No hay umbral automático de originalidad.")

In [ ]:
PAIRS = OUT / "neighbor_sheets"
PAIRS.mkdir(exist_ok=True)
for start in range(0,COUNT,20):
    page=Image.new("RGB",(4*296,5*172),"#eeeeee")
    draw=ImageDraw.Draw(page)
    for pos,record in enumerate(records[start:start+20]):
        x,y=(pos%4)*296,(pos//4)*172
        with Image.open(OUT / record["candidate_file"]) as im:
            page.paste(im.resize((128,128),Image.Resampling.NEAREST),(x+8,y+24))
        with Image.open(DATA / record["nearest_training_file"]) as im:
            page.paste(im.convert("RGB").resize((128,128),Image.Resampling.NEAREST),(x+152,y+24))
        draw.text((x+8,y+4),f"ID {record['candidate_id']:03d} GAN | vecino real",fill="black")
        draw.text((x+8,y+154),f"Distancia coseno: {record['cosine_distance']:.4f}",fill="black")
    page.save(PAIRS / f"neighbors_{start+1:03d}.png")
# Mostrar la primera página; las otras nueve quedan en Drive.
display(Image.open(PAIRS / "neighbors_001.png"))
print("LISTO: 200 candidatos y 10 páginas de vecinos guardadas en",OUT)
print("Revisa sheets/ y neighbor_sheets/. Luego comparte las 4 rejillas y los IDs que te gusten.")
print("Criterios: rostro reconocible, menos artefactos, diferencias en varios rasgos y sin copia casi idéntica del vecino.")
print("No hay todavía galería final; seleccionaremos 10 tras revisar esas evidencias.")